# Quantum Autoencoder (QAE) for Signal Processing and Noise Reduction

When dealing with high-frequency time-series or RF (Radio Frequency) signals, classical autoencoders compress information through a low-dimensional structural bottleneck. A Quantum Autoencoder does something unique: it takes data encoded into a broad register of qubits and trains a Parameterized Quantum Circuit (PQC) to compress the quantum state down into a smaller subset of qubits (the latent register or "bottleneck"), leaving the remaining qubits (the trash register) in a completely clean ground state ($|0\rangle$).

If a signal contains noise, that noise destroys the clean, low-entropy structure of the pure signal. By forcing the quantum system to pack all relevant information strictly into the latent qubits, the random, high-entropy noise components are naturally filtered out into the trash register.

## The Research Blueprint: Denoising Quantum Signal Processing
We will implement a Denoising Quantum Autoencoder. The model takes a synthetic 3-variable physical signal corrupted by severe environmental noise, embeds it into 3 qubits, compresses it down to 1 latent qubit, and measures how effectively it purifies the signal.


> Input State |ψ_noisy⟩ ───[  Encoder PQC (U)  ]───>  Latent Qubit  ───> [ Reconstructed Pure Signal ]
                                               └───>  Trash Qubits   ───> Measure against |00⟩



In [ ]:
# Install required packages (quiet mode for cleaner output)
!pip install --quiet pennylane
!pip install --quiet amazon-braket-pennylane-plugin

### Step 1: Generating the Noisy Signal Dataset

In [ ]:
import pennylane as qml
from pennylane import numpy as np

# 1. Synthesize a clean physical signal trajectory (e.g., three coupled physical components)
np.random.seed(42)
num_samples = 50

pure_signals = []
noisy_signals = []

for _ in range(num_samples):
    # Base phase component
    phi = np.random.uniform(0, np.pi)
    clean = np.array([np.sin(phi), np.cos(phi), np.sin(2*phi)])
    # Normalize to map safely to quantum state amplitudes
    clean /= np.linalg.norm(clean)

    # Inject severe localized noise to mimic an extreme communications channel
    noise = np.random.normal(0, 0.4, 3)
    noisy = clean + noise
    noisy /= np.linalg.norm(noisy)

    pure_signals.append(clean)
    noisy_signals.append(noisy)

print(f"Data ready. Sample raw signal features:\nPure:  {pure_signals[0]}\nNoisy: {noisy_signals[0]}")

### Step 2: Constructing the Quantum Autoencoder Circuit
We will use a 3-qubit register.
*   Qubit 0 will serve as our Latent Bottleneck.
*   Qubits 1 and 2 will act as our Trash Register.

The goal of the encoder circuit ($U$) is to compress all information into Qubit 0, meaning Qubits 1 and 2 should ideally finish in the exact state $|00\rangle$.

In [ ]:
num_qubits = 3
dev = qml.device("braket.local.qubit", wires=num_qubits)

def encoder_ansatz(weights):
    """A parameterized circuit acting as the spatial compressor."""
    # Layer 1 parameters
    for i in range(num_qubits):
        qml.RX(weights[0, i], wires=i)
        qml.RY(weights[1, i], wires=i)
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])

    # Layer 2 parameters
    for i in range(num_qubits):
        qml.RY(weights[2, i], wires=i)
    qml.CNOT(wires=[0, 2])

@qml.qnode(dev)
def autoencoder_cost_circuit(noisy_state, weights):
    # Step 1: Load the raw noisy signal vector via Amplitude Encoding
    qml.AmplitudeEmbedding(features=noisy_state, wires=range(num_qubits), pad_with=0.0)

    # Step 2: Execute the Encoder circuit transformation
    encoder_ansatz(weights)

    # Step 3: Measure the expectation value of the trash register (Wires 1 and 2)
    # If compression is perfect, Wires 1 and 2 collapse to |00>, yielding a combined identity score of 1.0
    return qml.expval(qml.Projector([0, 0], wires=[1, 2]))

### Step 3: Optimization Framework (Maximizing Compression Fidelity)

In [ ]:
# Initialize weights: 3 layers, 3 qubits layout
weights = np.random.uniform(0, 2 * np.pi, (3, num_qubits), requires_grad=True)
opt = qml.AdamOptimizer(stepsize=0.08)

def cost_fn(w):
    # We want to maximize the probability of measuring |00> on the trash qubits.
    # Therefore, we minimize (1.0 - Probability)
    total_loss = 0
    for i in range(len(noisy_signals)):
        prob_identity = autoencoder_cost_circuit(noisy_signals[i], w)
        total_loss += (1.0 - prob_identity)
    return total_loss / len(noisy_signals)

print("Training Quantum Signal Autoencoder...")
print("---------------------------------------")

for epoch in range(41):
    weights, loss = opt.step_and_cost(cost_fn, weights)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:2d} | Signal Reconstruction Loss: {loss:.5f}")

print("---------------------------------------")

### Step 4: Measuring Denoising Performance
To prove the autoencoder successfully separated signal from noise, we extract the structural feature maps and measure the overlap (State Fidelity) between our compressed reconstruction and the true, uncorrupted signal.

In [ ]:
# Evaluate the denoising capability on a test signal instance
test_noisy = noisy_signals[0]
test_pure = pure_signals[0]

@qml.qnode(dev)
def evaluate_denoised_state(noisy_state, weights):
    qml.AmplitudeEmbedding(features=noisy_state, wires=range(num_qubits), pad_with=0.0)
    encoder_ansatz(weights)
    return qml.state()

# Process state vectors
final_quantum_state = evaluate_denoised_state(test_noisy, weights)

# Calculate the inner product overlap against the target pure vector layout
pure_state_vector = np.zeros(2**num_qubits)
pure_state_vector[:3] = test_pure
pure_state_vector /= np.linalg.norm(pure_state_vector)

fidelity_before = np.abs(np.vdot(test_noisy, pure_state_vector))**2
# Trace out trash register impact conceptually via state overlap
fidelity_after = np.abs(np.vdot(final_quantum_state, pure_state_vector))**2

print("\n--- Signal Denoising Metrics ---")
print(f"Initial Noisy Signal Fidelity:  {fidelity_before*100:.2f}%")
print(f"Post-Compression Pure Fidelity: {fidelity_after*100:.2f}%")
print(f"Net Quantum SNR Improvement:    {((fidelity_after - fidelity_before) / fidelity_before)*100:+.2f}%")

## Directions for a Research Publication
If you want to frame this script as a foundational architecture for a research paper, look into these specific open areas:

**Quantum State Tomography via SWAP Test Restoration**
In the simple template above, we track the internal state vector using classical simulation boundaries. In a physical implementation on Amazon Braket hardware, you can't simply read out the state vector.
*   Research Pivot: Rebuild the decoding phase using a SWAP Test module. This uses an extra auxiliary qubit to measure the overlap between the reconstructed output and an actual target reference signal on hardware without requiring exponential classical computation.

**Quantum ConvNet Autoencoders (QCNN-AE) for Space Signals**
For deep-space communication or radar signal processing, signals arrive as massive, continuous high-dimensional waveforms. Standard all-to-all PQCs face the problem of Barren Plateaus (vanishing gradients) as the time-series grid expands.
*   Research Pivot: Introduce a Quantum Convolutional Neural Network (QCNN) architecture inside the encoder loop. By combining localized quantum convolutions with a stride-based pooling operation that drops "trash" qubits sequentially across layers, your students can demonstrate a model that scales logarithmically ($O(\log N)$ parameters) to circumvent barren plateaus.